usage see peters/GlitchOnHs2Test.ipynb

In [2]:
%matplotlib widget
import chipwhisperer as cw
import numpy as np
import matplotlib.pyplot as plt
import time

LA_NAMES = {
    "glitch": [
        "glitch output",
        "glitch src clock",
        "MMCM1 (offset) out",
        "MMCM2 (width) out",
        "glitch go",
        "capture trigger",
        "glitch enable",
        "manual trig (src clk)",
        "trig in MMCM1 clk",
    ],
    "CW 20-pin": ["IO1", "IO2", "IO3", "IO4", "HS1", "HS2",
                  "AUX MCX", "TRIG MCX", "ADC sampling clock"],
    "USERIO 20-pin": ["D0", "D1", "D2", "D3", "D4", "D5", "D6", "D7", "CK"],
}

def read_capture_data(scope):
    """Read the captured logic analyzer (LA) data and split it into single signals.

    Call this after scope.LA.arm() and after the event you want to capture
    has happened. It uses the LA capture group that is currently set.

    Returns a dict with:
        time_axis_s    : numpy array of sample times in seconds, starting at 0
        sigs           : list of 0/1 arrays, one per signal of the capture group,
                         in the same order as LA_NAMES[capture_group]
        names          : list of signal names of the capture group (same order as sigs)
    """
    la = scope.LA

    # Read the raw capture buffer from the FPGA.
    # check_empty=False: do not raise an error if the buffer looks empty,
    # so the caller gets an all-zero capture instead of an exception.
    raw = la.read_capture_data(check_empty=False)

    # Signal names of the current capture group, e.g. "IO1", "HS2", ...
    names = list(LA_NAMES[la.capture_group])

    # The raw data packs all signals of the capture group into the same samples.
    # extract(raw, i) pulls out signal i as its own array.
    sigs = [la.extract(raw, i) for i in range(len(names))]

    # Effective sample rate in Hz: base LA sampling clock divided by the downsample factor.
    f_la = la.sampling_clock_frequency / la.downsample

    # Time axis in seconds: sample index divided by the sample rate.
    # All signals have the same length, so sigs[0] is used for the length.
    res = dict(
        time_axis_s=np.arange(len(sigs[0])) / f_la,
        sigs=sigs,
        names=names,
    )
    return res


# Time units in seconds, ordered from large to small (used for auto selection)
_TIME_UNITS = (("s", 1.0), ("ms", 1e-3), ("us", 1e-6), ("ns", 1e-9))


def auto_time_unit(span_s):
    """Return the best time unit for a time span given in seconds.

    The largest unit is chosen in which the span is still >= 1, e.g.
    2.5e-6 s -> "us", 400e-9 s -> "ns", 0.02 s -> "ms".
    Spans <= 0 or smaller than 1 ns fall back to "ns".
    """
    for name, scale in _TIME_UNITS:
        if span_s >= scale:
            return name
    return "ns"


def plot_la_interactive(res, unit="auto", snap=True, title="", channels=None):
    """Plot LA signals as stacked digital traces with two measurement cursors.

    Parameters
    ----------
    res      : dict from read_capture_data() with the keys
               time_axis_s (seconds), sigs (list of 0/1 arrays), names (list of str).
               A dict from run_glitch() also works if it has a "names" entry
               or if you pass `channels` as indices.
    unit     : time unit of the x axis: "auto", "s", "ms", "us" or "ns".
               "auto" (default) is calculated from the length of the capture.
    snap     : if True, a clicked cursor jumps to the nearest signal edge.
    title    : optional text shown above the plot.
    channels : None -> plot all signals,
               or a list of indices / names, e.g. [0, 3] or ["IO3", "IO4"].
               The order of the list is the order of the rows in the plot.

    Usage: left-click sets cursor A, then B (prints A, B, delta t and 1/delta t),
    a third click starts a new measurement, right-click clears the cursors.
    Cursors are inactive while the zoom/pan tool of the toolbar is selected.
    Returns the matplotlib figure.
    """
    # Time axis in seconds, as delivered by read_capture_data()
    t_s = np.asarray(res["time_axis_s"], dtype=float)

    # Resolve the unit: "auto" is derived from the total capture length
    if unit == "auto":
        span_s = float(t_s.max() - t_s.min()) if t_s.size else 0.0
        unit = auto_time_unit(span_s)

    # Conversion factor: unit -> seconds
    scale = dict(_TIME_UNITS)[unit]

    sigs_all = res["sigs"]
    # Names come from the result; fall back to "ch0", "ch1", ... if they are missing
    names_all = res.get("names") or [f"ch{i}" for i in range(len(sigs_all))]

    # Time axis converted from seconds to the chosen unit
    t = t_s / scale

    # Select channels: names are converted to indices, indices are kept
    if channels is None:
        idx = list(range(len(sigs_all)))
    else:
        idx = [names_all.index(c) if isinstance(c, str) else int(c) for c in channels]
    names = [names_all[i] for i in idx]
    sigs = [sigs_all[i] for i in idx]

    # One subplot row per channel, all sharing the same time axis
    n_ch = len(sigs)
    fig, axes = plt.subplots(n_ch, 1, figsize=(11, 1 + 0.7 * n_ch), sharex=True, squeeze=False)
    axs = axes[:, 0]

    # Draw every signal as a step plot and collect all edge times (for cursor snapping)
    edge_times = []
    for ax, name, sig in zip(axs, names, sigs):
        sig = np.asarray(sig)
        ax.step(t, sig, where="post")
        ax.set_ylabel(name, rotation=0, ha="right", va="center", fontsize=8)
        ax.set_yticks([0, 1])
        ax.set_ylim(-0.2, 1.2)
        ax.grid(True, alpha=0.3)
        # Index i+1 is the first sample after a change between sample i and i+1
        edge_times.append(t[np.flatnonzero(np.diff(sig) != 0) + 1])
    # Sorted list of all edge times over all channels, used for snapping
    edges = np.sort(np.concatenate(edge_times))

    # The x label shows the unit that was chosen (or calculated)
    axs[-1].set_xlabel(f"time ({unit})")
    help_txt = "Left-click: cursor A/B | Right-click: clear"
    fig.suptitle(f"{title}\n{help_txt}" if title else help_txt, fontsize=9)

    # State of the cursors: x positions (marks) and the matplotlib line objects
    marks, lines = [], []

    def clear():
        """Remove both cursors from the plot and reset the state."""
        for ln in lines:
            ln.remove()
        lines.clear()
        marks.clear()

    def on_click(event):
        """Mouse handler: left click sets a cursor, right click clears them."""
        # Ignore clicks while the zoom/pan tool is active
        tb = fig.canvas.toolbar
        if tb is not None and getattr(tb, "mode", "") != "":
            return
        # Ignore clicks outside the plot area
        if event.inaxes not in axs:
            return
        # Right click: remove cursors and restore the help text
        if event.button == 3:
            clear()
            fig.suptitle(f"{title}\n{help_txt}" if title else help_txt, fontsize=9)
            fig.canvas.draw_idle()
            return
        # Only the left button sets cursors
        if event.button != 1:
            return

        # Clicked x position, optionally snapped to the nearest signal edge
        x = event.xdata
        if snap and edges.size:
            x = edges[np.argmin(np.abs(edges - x))]

        # A third click starts a new measurement
        if len(marks) == 2:
            clear()
        marks.append(x)

        # Cursor A is red, cursor B is green; the line spans all channels
        color = "tab:red" if len(marks) == 1 else "tab:green"
        for ax in axs:
            lines.append(ax.axvline(x, color=color, linestyle="--", lw=1))

        # Show the cursor positions; after the second click also delta t and 1/delta t
        if len(marks) == 1:
            msg = f"A = {x:.2f} {unit}"
        else:
            dt = abs(marks[1] - marks[0])                         # in the chosen unit
            f = 1 / (dt * scale) if dt > 0 else float("nan")      # frequency in Hz
            msg = (f"A = {marks:.2f} {unit}   B = {marks:.2f} {unit}   "
                   f"Δt = {dt:.2f} {unit}   1/Δt = {f/1e6:.3f} MHz")
            
        fig.suptitle(msg, fontsize=10)
        fig.canvas.draw_idle()

    fig.canvas.mpl_connect("button_press_event", on_click)
    fig.tight_layout()
    plt.show()
    return fig

def calc_glitch_times(scope, latency_ns=0):
    """Convert glitch settings to times in ns (computed, not measured).

    latency_ns: fixed trigger->output latency you measured 
    """
    
    phase_shift_steps = scope.glitch.phase_shift_steps
    ext_offset = scope.glitch.ext_offset
    offset = scope.glitch.offset
    width = scope.glitch.width
    repeat = scope.glitch.repeat
    output = scope.glitch.output
    f_glitch=scope.clock.clkgen_freq
    
    T = 1 / f_glitch  # period in s
    ns = T * 1e9      # period in ns
    d = dict(
        T_ns=ns,
        phase_shift_steps=phase_shift_steps,
        ext_offset=ext_offset,
        ext_offset_ns=ext_offset * ns,            # whole clock cycles after the trigger
        offset=offset,
        offset_ns=offset / phase_shift_steps * ns,            # fine phase shift of the glitch position
        width=width,
        width_ns=width / phase_shift_steps * ns,              # fine phase shift: pulse width (non-enable_only modes)
        repeat=repeat,
        enable_ns=repeat * ns,                          # length of the enable window, repeat cycles
    )
    d["start_ns"] = d["ext_offset_ns"] + d["offset_ns"] + latency_ns
    if output == "enable_only":
        d["pulse_ns"] = d["enable_ns"]            # width/offset do not change this output
    else:
        d["pulse_ns"] = d["width_ns"]             # per glitch cycle, if my reading of the mode is right
    return d

def fmt_ns(x):
    if x != x:                      # NaN
        return "n/a"
    if x >= 100:
        return f"{x:.0f} ns"
    if x >= 1:
        return f"{x:.1f} ns"
    return f"{x * 1000:.0f} ps"
    
def caption(res):
    """Text with the computed times and, if found, the measured ones."""
    s = (f"1/clkgen_freq={res['T_ns']} ns phase_shift_steps={res['phase_shift_steps']}\n"
     f"ext_offset={res['ext_offset']} ({fmt_ns(res['ext_offset_ns'])})  "
     f"offset={res['offset']} ({fmt_ns(res['offset_ns'])})  "
     f"width={res['width']} ({fmt_ns(res['width_ns'])})  "
     f"repeat={res['repeat']} (pulse {fmt_ns(res.get('pulse_ns', float('nan')))})")

#    s += f"\ncomputed start={tm['start_ns']:.0f} ns"
#    if "delay_ns" in res:
#        s += f"   measured: delay={res['delay_ns']:.0f} ns"
#        if "pulse_ns" in res:
#            s += f"  pulse={res['pulse_ns']:.0f} ns"
#        diff = res['delay_ns'] - tm['start_ns']
#        s += f"  start diff={diff:.0f} ns"
    return s
